In [1]:
# --- IMPORTS & CONFIGURATION ---
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Plotting style preferences
plt.rcParams['figure.figsize'] = (10, 8)
sns.set_style("whitegrid")

In [2]:
# --- DATA LOADING ---
# NOTE: Paths updated to absolute paths for the original data location
base_dir = '/ceph/MethDev/pbio'

# Load Methylation Data
mcds_path = f'{base_dir}/All_mcds.mcds'
# (Assuming MCDS loading logic is handled by a library or previous step not shown, 
# but keeping the path definition here as in your original script)

# Load RNA/Metadata
# Corrected paths to point to the /ceph/ location
meta = pd.read_csv(f'{base_dir}/data/Arab6_cells_and_clusters.csv')
rna_matrix = pd.read_csv(f'{base_dir}/novaseq_demux/count.csv', index_col=0).T

In [3]:
meta

,Cell,Cluster
0,NJ_221007_P1.2.K15.J3,1
1,NJ_221007_P1.2.K15.E16,12
2,NJ_221007_P1.2.K15.C15,4
3,NJ_221007_P1.2.K15.J4,1
4,NJ_221007_P1.2.K15.M16,4
...,...,...
3317,JW_240119_Arab_P.4.I3.H7,1
3318,JW_240119_Arab_P.4.I3.I7,1
3319,JW_240119_Arab_P.4.I3.P8,0
3320,JW_240119_Arab_P.4.I3.E8,0


In [4]:
rna_matrix

,AT1G01010,AT1G01020,AT1G01030,AT1G01040,AT1G01046,AT1G01050,AT1G01060,AT1G01070,AT1G01080,AT1G01090,...,ATMG01380,ATMG01390,ATMG01400,ATMG01410,ATMG09450,ATMG09730,ATMG09740,ATMG09950,ATMG09960,ATMG09980
NJ_221007_P1-2-K15-J3,0,0,0,44,0,69,0,0,0,0,...,0,20,0,0,0,0,0,0,0,0
NJ_221007_P1-2-K15-E16,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NJ_221007_P1-2-K15-C15,0,0,0,3,0,110,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NJ_221007_P1-2-K15-J4,0,78,0,623,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NJ_221007_P1-2-K15-M16,0,3,0,36,0,28,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NJ_221007_P8-5-G10-N10,0,6,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NJ_221007_P8-5-G10-F22,0,2,0,1,0,0,15,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NJ_221007_P8-5-G10-O10,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NJ_221007_P8-5-G10-N21,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
# --- DATA PREPROCESSING ---

# 1. Prepare Metadata
# Fix: The column is named 'Cell', not 'cell_id'
# Fix: Apply the string replacement found in your original script (Cell 6)
meta['Cell'] = meta['Cell'].str.replace('.', '-')
meta = meta.set_index('Cell')

# 2. Cluster Merging
# Merging specific sub-clusters into main groups
# Cluster 6, 0, 4 -> Merged into Cluster 1; Cluster 5 -> Merged into Cluster 3
meta['Cluster'] = meta['Cluster'].replace({6: 1, 0: 1, 4: 1, 5: 3})

# NOTE: Clusters 2 and 9 are PRESERVED in this version.

# 3. Align RNA and Methylation Metadata
# Ensure the RNA matrix only contains cells present in our metadata
# Use the index (which is now the Cell ID) for intersection
common_cells = meta.index
rna_matrix = rna_matrix.loc[rna_matrix.index.intersection(common_cells)]

In [ ]:
# --- VISUALIZATION ---

# Heatmap generation
plt.figure(figsize=(12, 10))
sns.heatmap(
    rna_matrix.corr(),  # Or your specific correlation matrix variable
    cmap='viridis',
    annot=False
)
plt.title("Methylation vs Expression Correlation (All Clusters)")
plt.show()